In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 07 · A2A: Agent Cards and identity between agents — practice

    **Primer section:** §7.2. Build and sign a card, catch tampering, and implement the two sides of a
    delegated hop.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import json

import jwt  # display only

from agentsec.a2a import (
    A2AAuthError,
    authorize_inbound,
    build_agent_card,
    card_from_dict,
    card_to_dict,
    required_scopes,
    sign_agent_card,
    token_for_peer,
    verify_agent_card,
)
from agentsec.identity import AgentIdentity, LocalRuntimeCA, TokenIssuer, UserPrincipal


def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
support = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
refunds = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="refunds-agent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

issuer = TokenIssuer()
ca = LocalRuntimeCA()
support_cert = ca.issue(support)

SUPPORT_URL = "https://agents.acme.example/support"
REFUNDS_URL = "https://agents.acme.example/refunds"
FRONT_END = "https://app.acme.example"

## Exercise 1 — build the card and read its requirements

Build the support agent's card for `SUPPORT_URL` and confirm the bearer scheme and the required
scope.

In [ ]:
card = build_agent_card(agent=____, name="support", url=____, issuer=issuer.issuer)
doc = card_to_dict(card)

assert card.security_schemes["bearer"].http_auth_security_scheme.scheme == "bearer"
assert ____(card) == {"agent:invoke"}
assert doc["supportedInterfaces"][0]["url"] == SUPPORT_URL
print(json.dumps(doc["securityRequirements"], indent=2))

## Exercise 2 — sign, verify, and detect tampering

Sign the card with the support agent's certificate key (use the thumbprint as `kid`), verify it,
then implement `is_authentic(doc)` that returns whether a card *document* (dict) verifies against
the trusted keys. It must reject a tampered URL and an unsigned card.

In [ ]:
signed = sign_agent_card(card, ____, kid=____)
trusted_keys = {support_cert.thumbprint: support_cert.private_key.public_key()}

def is_authentic(doc: dict) -> bool:
    raise NotImplementedError("fill me")  # verify_agent_card(card_from_dict(doc), trusted_keys) → True/False

stored = card_to_dict(signed)
tampered = json.loads(json.dumps(stored))
tampered["supportedInterfaces"][0]["url"] = "https://evil.example/support"

assert verify_agent_card(signed, trusted_keys) == support_cert.thumbprint
assert is_authentic(stored)
assert not is_authentic(tampered)
assert not is_authentic(card_to_dict(card))
print("signed card verifies; tampered and unsigned cards do not")

## Exercise 3 — the caller's side: a token for the peer

Ana's login token has scopes `agent:invoke tickets:read`. Obtain the token the support agent should
present to the refunds agent and check its audience, actor and scope.

In [ ]:
user_token = issuer.mint(subject=ana.subject, audience=FRONT_END, scope="agent:invoke tickets:read", extra={"email": ana.email})
peer_token = token_for_peer(issuer, caller=____, current_token=____, peer_audience=____)

claims = peek(peer_token)
assert claims["aud"] == REFUNDS_URL
assert claims["sub"] == "u-ana" and claims["act"]["sub"] == support.spiffe_id
assert claims["scope"] == "agent:invoke"
print("peer token:", {k: claims[k] for k in ("sub", "aud", "scope")}, "act:", short(claims["act"]["sub"]))

## Exercise 4 — the receiver's side: authorize the hop

Authorize the inbound request at the refunds agent, then show that a forwarded user token and a
depth-0 policy are both rejected.

In [ ]:
claims, authority = authorize_inbound({"Authorization": f"Bearer {peer_token}"}, issuer=issuer, audience=____, this_agent=____)
assert authority.user.email == "ana@customer.example"
assert authority.chain == (refunds.spiffe_id, support.spiffe_id)
assert authority.scopes == {"agent:invoke"}

def rejected(token: str, **kw) -> bool:
    try:
        authorize_inbound({"Authorization": f"Bearer {token}"}, issuer=issuer, audience=REFUNDS_URL, this_agent=refunds, **kw)
        return False
    except ____:
        return True

assert rejected(user_token)                              # forwarded: wrong audience
assert rejected(peer_token, max_delegation_depth=0)      # chain too deep
assert not rejected(token_for_peer(issuer, caller=support, current_token=None, peer_audience=REFUNDS_URL))  # own authority
print("hop chain:", [short(a) for a in authority.chain])

## Exercise 5 — a second hop

The refunds agent calls a payments agent at `PAYMENTS_URL`. Obtain the token and authorize it at the
payments agent; the actor chain must be `[refunds, support]` and the hop chain `(payments, refunds, support)`.

In [ ]:
payments = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="payments-agent", org_id=ORG)
PAYMENTS_URL = "https://agents.acme.example/payments"

hop2_token = token_for_peer(issuer, caller=____, current_token=____, peer_audience=PAYMENTS_URL)
claims2, authority2 = authorize_inbound({"Authorization": f"Bearer {hop2_token}"}, issuer=issuer, audience=____, this_agent=____)

assert claims2.actor_chain == [refunds.spiffe_id, support.spiffe_id]
assert authority2.chain == (payments.spiffe_id, refunds.spiffe_id, support.spiffe_id)
assert claims2.subject == "u-ana"
print("actor chain:", [short(a) for a in claims2.actor_chain])

**In one sentence:** "Cards declare and sign; hops re-authorize; delegation is exchanged not
forwarded, so the chain is explicit and scopes only narrow."